In [ ]:
# ============================================================
# PURPOSE: Initialize Spark Session for GBT stock forecasting
# ============================================================

# Import SparkSession — the main entry point for working with Spark SQL/DataFrames
from pyspark.sql import SparkSession

# Create or retrieve an existing Spark session.
# .appName() sets the application name shown in Spark UI.
spark = SparkSession.builder \
    .appName("StockPricePrediction_GBT_Mongo") \
    .getOrCreate()

# Confirm Spark startup
print("Spark Session Started")


In [ ]:
# ============================================================
# PURPOSE: Load all cleaned CSV stock files into one Spark DataFrame
# ============================================================

import os
from pyspark.sql.functions import col

# Folder containing all cleaned CSV files
clean_folder = "/Users/venubandi/Desktop/DATA_603_/cleaned_csvs/"

# List all .csv filenames in the folder with full path
clean_files = [clean_folder + f for f in os.listdir(clean_folder) if f.endswith(".csv")]

# Initialize master DataFrame and counter
df = None
total_rows = 0

# ------------------------------------------------------------
# Loop through each cleaned CSV and append to master DataFrame
# ------------------------------------------------------------
for f in clean_files:
    print("Loading:", f)

    # Load CSV into temporary Spark DataFrame
    df_temp = spark.read.csv(f, header=True, inferSchema=True)

    # First file -> assign directly, else union with existing DF
    df = df_temp if df is None else df.unionByName(df_temp)

    # Count rows loaded from this file
    total_rows += df_temp.count()

# ------------------------------------------------------------
# Sort final combined DataFrame by Ticker and Date for consistency
# ------------------------------------------------------------
df = df.orderBy("Ticker", "Date")

# Print summary
print("Loaded master dataframe:", total_rows)
df.printSchema()


In [ ]:
# ============================================================
# Create Regression Target Column: Next Day Close Price
# ============================================================

from pyspark.sql.window import Window
from pyspark.sql.functions import lead

# Window specification: for each ticker, order rows by date
w = Window.partitionBy("Ticker").orderBy("Date")

# Add "Next_Close" = next day's closing price (shifted by 1)
df_reg = df.withColumn(
    "Next_Close",
    lead("Close", 1).over(w)
)

# Remove final rows per ticker where Next_Close is null
df_reg = df_reg.dropna(subset=["Next_Close"])

df_reg.show(10, truncate=False)


In [ ]:
# ============================================================
# Feature Engineering: Returns, Moving Averages, Volatility, etc.
# ============================================================

from pyspark.sql.window import Window
import pyspark.sql.functions as F

# Window specification: compute rolling stats per ticker ordered by date
w = Window.partitionBy("Ticker").orderBy("Date")

# Create feature columns
df_feat = (
    df_reg
    # Daily return
    .withColumn(
        "Return",
        (F.col("Close") - F.lag("Close").over(w)) / F.lag("Close").over(w)
    )
    # 7-day moving average of Close
    .withColumn(
        "MA_7",
        F.avg("Close").over(w.rowsBetween(-6, 0))
    )
    # 30-day moving average of Close
    .withColumn(
        "MA_30",
        F.avg("Close").over(w.rowsBetween(-29, 0))
    )
    # 30-day rolling volatility (std dev of Close)
    .withColumn(
        "Volatility_30",
        F.stddev("Close").over(w.rowsBetween(-29, 0))
    )
    # Daily range = High − Low
    .withColumn(
        "Range",
        F.col("High") - F.col("Low")
    )
    # Volume change from previous day
    .withColumn(
        "Volume_Change",
        F.col("Volume") - F.lag("Volume").over(w)
    )
)

# Remove rows with incomplete feature values
df_feat = df_feat.dropna()

df_feat.show(10, truncate=False)


In [ ]:
# ============================================================
# Prepare Final ML Dataset (Select Features + Drop Missing)
# ============================================================

# List of engineered feature columns that will be used for training
feature_cols = [
    "Close", "Return", "MA_7", "MA_30",
    "Volatility_30", "Range", "Volume", "Volume_Change"
]

# Select final modeling columns:
#  - Ticker, Date → metadata
#  - Next_Close   → label for regression
#  - feature_cols → model input features
ml_df = (
    df_feat
    .select(["Ticker", "Date", "Next_Close"] + feature_cols)
    .dropna()   # Ensure full rows for ML training
)

print("→ Final ML dataset row count:", ml_df.count())
ml_df.show(10, truncate=False)


In [ ]:
# ============================================================
# VectorAssembler : Convert Feature Columns into ML Vectors
# ============================================================

from pyspark.ml.feature import VectorAssembler

# Create a VectorAssembler to combine all feature columns
assembler = VectorAssembler(
    inputCols=feature_cols,   # columns used as model inputs
    outputCol="features"      # output vector column
)

# Transform ML dataframe → add "features" vector column
df_final = (
    assembler.transform(ml_df)
             .select("Ticker", "Date", "Next_Close", "features")
)

print(" Assembled 'features' column successfully.")
df_final.show(5, truncate=False)


In [ ]:
# ============================================================
# Train–Test Split (80% Training, 20% Testing)
# ============================================================

# Randomly split the data into training and testing sets
train_df, test_df = df_final.randomSplit([0.8, 0.2], seed=42)

# Print dataset sizes for verification
print(" Train rows:", train_df.count())
print(" Test rows:", test_df.count())


In [ ]:
# ============================================================
# Train Gradient-Boosted Tree (GBT) Regression Model
# ============================================================

from pyspark.ml.regression import GBTRegressor

# Initialize the GBT Regressor with tuned hyperparameters
gbt = GBTRegressor(
    labelCol="Next_Close",
    featuresCol="features",
    maxDepth=5,
    maxIter=50,
    stepSize=0.1,
    seed=42
)

# Train the model on the training dataset
gbt_model = gbt.fit(train_df)

print(" GBT Regression model trained successfully!")


In [ ]:
# ============================================================
# Evaluate GBT Model Performance (RMSE & R²)
# ============================================================

from pyspark.ml.evaluation import RegressionEvaluator

# Generate predictions on training and testing datasets
pred_train = gbt_model.transform(train_df)
pred_test  = gbt_model.transform(test_df)

# Initialize evaluator for RMSE and R²
ev = RegressionEvaluator(
    labelCol="Next_Close",
    predictionCol="prediction"
)

# ---- Evaluate RMSE ----
train_rmse = ev.evaluate(pred_train, {ev.metricName: "rmse"})
test_rmse  = ev.evaluate(pred_test,  {ev.metricName: "rmse"})

# ---- Evaluate R² ----
train_r2 = ev.evaluate(pred_train, {ev.metricName: "r2"})
test_r2  = ev.evaluate(pred_test,  {ev.metricName: "r2"})

# Print results
print("Train RMSE:", train_rmse)
print("Test RMSE :", test_rmse)
print("Train R²  :", train_r2)
print("Test R²   :", test_r2)


In [ ]:
# ==============================================================
# PURPOSE: Create predictions_df (Actual + Predicted values)
# ==============================================================

print(" Generating predictions_df ...")

predictions_df = (
    gbt_model.transform(df_final)
    .select("Ticker", "Date", "Next_Close", F.col("prediction"))
)

predictions_df.show(10)
print(" predictions_df created successfully!")


In [ ]:
# ==============================================================
# PURPOSE: Save predictions_df into MongoDB safely
# ==============================================================

from pymongo import MongoClient
import pandas as pd

# -----------------------------
# Connect to MongoDB
# -----------------------------
client = MongoClient("mongodb://localhost:27017/")
db = client["stock_prediction_db"]
predictions_col = db["predictions"]

# -----------------------------
# Convert Spark DF -> Pandas
# -----------------------------
pdf = predictions_df.toPandas()

# Rename the prediction column for clarity
pdf.rename(columns={"prediction": "Predicted_Close"}, inplace=True)

# -----------------------------
# FIX: Convert Date into ISO string or datetime
# MongoDB cannot store datetime.date -> must convert
# -----------------------------
pdf["Date"] = pd.to_datetime(pdf["Date"])        # convert to datetime64[ns]
pdf["Date"] = pdf["Date"].dt.strftime("%Y-%m-%d")  # store as string (recommended)

# -----------------------------
# Convert to Python dictionaries
# -----------------------------
records = pdf.to_dict("records")

# -----------------------------
# Insert into Mongo
# -----------------------------
predictions_col.drop()  # optional -> clear old data
predictions_col.insert_many(records)

print("$ predictions_df successfully saved into MongoDB!")
print(f"Total records inserted: {len(records)}")


In [ ]:
# ============================================================
# PURPOSE: Plot DAILY actual vs predicted values for each ticker
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd

tickers = predictions_df.select("Ticker").distinct().rdd.flatMap(lambda x: x).collect()

for ticker in tickers:
    print(f"\n=== Plotting {ticker} ===")
    
    pdf = predictions_df.filter(col("Ticker") == ticker).toPandas()
    pdf["Date"] = pd.to_datetime(pdf["Date"])
    pdf = pdf.sort_values("Date")

    plt.figure(figsize=(14, 5))
    plt.plot(pdf["Date"], pdf["Next_Close"], label="Actual Close", linewidth=2)
    plt.plot(pdf["Date"], pdf["prediction"], label="Predicted Close", linewidth=2)
    plt.title(f"Daily Actual vs Predicted Close Price — {ticker}")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
# ======================================================
# FULL STOCK DASHBOARD WITH CLEAN SUMMARY + MONGO LOGGING
# ======================================================

import gradio as gr
import yfinance as yf
import pandas as pd
import plotly.express as px
from datetime import timedelta, datetime
import pymongo
import numpy as np


# ======================================================
# 0 — MONGO DB CLIENT
# ======================================================
MONGO_URI = "mongodb://localhost:27017/"
mongo_client = pymongo.MongoClient(MONGO_URI)
db = mongo_client["StockProject"]
log_collection = db["forecast_logs"]


# ======================================================
# 1 — TICKER LIST FROM SPARK DF
# ======================================================
TICKER_LIST = (
    predictions_df
    .select("Ticker")
    .distinct()
    .rdd.flatMap(lambda x: x)
    .collect()
)
TICKER_LIST = sorted(TICKER_LIST)


# ======================================================
# 2 — YAHOO FINANCE TREND VIEWER (TAB 1)
# ======================================================
def load_data(tickers, window):
    if len(tickers) == 0:
        return "Please select at least one ticker."

    mapping = {
        "Daily": ("1d", "1y"),
        "Weekly": ("1wk", "5y"),
        "Monthly": ("1mo", "10y"),
        "Yearly": ("3mo", "max"),
        "Full History": ("1d", "max")
    }

    interval, period = mapping[window]

    data = yf.download(
        tickers=tickers,
        interval=interval,
        period=period,
        auto_adjust=True
    )

    if isinstance(data.columns, pd.MultiIndex):
        data = data["Close"]

    data.dropna(how="all", inplace=True)

    fig = px.line(data, title=f"{window} Prices for: {', '.join(tickers)}")
    fig.update_layout(height=500)

    return fig


# ======================================================
# 3 — FORECAST ENGINE (NaN-SAFE)
# ======================================================
def _simulate_forecast_single(ticker, n_days):
    sdf_hist = (
        df_feat.filter(df_feat.Ticker == ticker)
        .select("Date", "Close", "Return", "MA_7", "MA_30",
                "Volatility_30", "Range", "Volume", "Volume_Change")
        .orderBy("Date")
    )

    pdf_hist = sdf_hist.toPandas()
    if pdf_hist.empty:
        return None, None

    pdf_hist["Date"] = pd.to_datetime(pdf_hist["Date"])
    base_df = pdf_hist.copy()
    future_rows = []

    for _ in range(n_days):
        last = base_df.iloc[-1]

        feature_dict = {}
        for col in feature_cols:
            v = float(last[col]) if col in last else 0.0
            if pd.isna(v) or np.isinf(v):
                v = 0.0
            feature_dict[col] = v

        feature_dict["Ticker"] = ticker
        feature_dict["Date"] = last["Date"]

        sdf = spark.createDataFrame([feature_dict])
        sdf_feat = assembler.transform(sdf).select("features")

        pred_value = (
            gbt_model.transform(sdf_feat)
            .select("prediction")
            .collect()[0]["prediction"]
        )

        new_close = float(pred_value)
        new_date = last["Date"] + timedelta(days=1)

        prev_close = float(last["Close"])
        new_return = (new_close - prev_close) / prev_close if prev_close != 0 else 0

        # Rolling metrics
        closes = list(base_df["Close"]) + [new_close]
        ma7 = pd.Series(closes).rolling(7, min_periods=1).mean().iloc[-1]
        ma30 = pd.Series(closes).rolling(30, min_periods=1).mean().iloc[-1]

        rets = list(base_df["Return"].fillna(0)) + [new_return]
        vol30 = pd.Series(rets).rolling(30, min_periods=2).std().fillna(0).iloc[-1]

        def clean(x): return 0.0 if pd.isna(x) or np.isinf(x) else float(x)

        new_row = {
            "Date": new_date,
            "Close": clean(new_close),
            "Return": clean(new_return),
            "MA_7": clean(ma7),
            "MA_30": clean(ma30),
            "Volatility_30": clean(vol30),
            "Range": clean(last["Range"]),
            "Volume": clean(last["Volume"]),
            "Volume_Change": 0.0
        }

        base_df = pd.concat([base_df, pd.DataFrame([new_row])], ignore_index=True)
        future_rows.append({"Date": new_date, "Predicted_Close": new_close})

    return pdf_hist, pd.DataFrame(future_rows)


# ======================================================
# 4 — BUY / HOLD / SELL WITH CLEAN SUMMARY + MONGO LOGGING
# ======================================================
def compare_companies(tickers, horizon_label):

    if not tickers:
        return None, None, "No tickers selected."

    horizon_map = {"Next 7 days": 7, "Next 10 days": 10, "Next 15 days": 15}
    n_days = horizon_map[horizon_label]

    rows = []

    # ---------------- RUN FORECAST FOR EACH TICKER ----------------
    for ticker in tickers:
        hist, fc = _simulate_forecast_single(ticker, n_days)
        if hist is None or fc is None:
            continue

        last_actual = float(hist["Close"].iloc[-1])
        last_pred = float(fc["Predicted_Close"].iloc[-1])
        ret_pct = ((last_pred - last_actual) / last_actual) * 100

        if ret_pct >= 5:
            signal = "BUY"
        elif ret_pct <= -2:
            signal = "SELL"
        else:
            signal = "HOLD"

        rows.append({
            "Ticker": ticker,
            "Predicted_Return_%": ret_pct,
            "Signal": signal
        })

        # -------- LOG TICKER-LEVEL ENTRY INTO MONGO --------
        log_collection.insert_one({
            "Type": "TickerLevel",
            "Ticker": ticker,
            "HorizonDays": n_days,
            "PredictedReturn": ret_pct,
            "Signal": signal,
            "ActualClose": last_actual,
            "PredictedClose": last_pred,
            "Timestamp": datetime.utcnow()
        })

    df = pd.DataFrame(rows).sort_values("Predicted_Return_%", ascending=False)

    # ---------------- CHART ----------------
    fig = px.bar(
        df, x="Ticker", y="Predicted_Return_%",
        color="Signal", text="Signal",
        title=f"Predicted Returns — {horizon_label}"
    )

    # ======================================================
    #           CLEAN BEAUTIFUL SUMMARY
    # ======================================================
    summary_parts = []

    # BUY
    buy_df = df[df["Signal"] == "BUY"]
    if not buy_df.empty:
        part = "🔥 $ BUY Recommendations\n"
        for _, row in buy_df.iterrows():
            part += (
                f"• {row['Ticker']} → Forecasted to rise {row['Predicted_Return_%']:.2f}%. "
                "Strong upward momentum and improving technical signals suggest a favorable entry opportunity.\n"
            )
        summary_parts.append(part)
    else:
        summary_parts.append("🔥 $ BUY Recommendations\n• No BUY opportunities detected.\n")

    # SELL
    sell_df = df[df["Signal"] == "SELL"]
    if not sell_df.empty:
        part = "\n🛑 - SELL Recommendations\n"
        for _, row in sell_df.iterrows():
            part += (
                f"• {row['Ticker']} → Expected to fall {abs(row['Predicted_Return_%']):.2f}%. "
                "Downward pressure and weakening price structure suggest reducing exposure to avoid potential losses.\n"
            )
        summary_parts.append(part)
    else:
        summary_parts.append("\n🛑 - SELL Recommendations\n• No SELL alerts detected.\n")

    # HOLD
    hold_df = df[df["Signal"] == "HOLD"]
    if not hold_df.empty:
        part = "\n🤝 HOLD Recommendations\n"
        for _, row in hold_df.iterrows():
            part += (
                f"• {row['Ticker']} → Expected movement **{row['Predicted_Return_%']:.2f}%. "
                "Price outlook stable — neither strong gain nor loss expected.\n"
            )
        summary_parts.append(part)
    else:
        summary_parts.append("\n🤝 HOLD Recommendations\n• No HOLD recommendations at this time.\n")

    summary = "\n".join(summary_parts)

    # -------- LOG SUMMARY INTO MONGO --------
    log_collection.insert_one({
        "Type": "SummaryLevel",
        "RequestTickers": tickers,
        "Horizon": horizon_label,
        "SummaryText": summary,
        "Timestamp": datetime.utcnow()
    })

    return fig, df, summary


# ======================================================
# 5 — MONGO LOG VIEWER (TAB 3)
# ======================================================
def view_logs(selected_ticker):
    query = {} if selected_ticker == "All" else {"Ticker": selected_ticker}

    logs = list(log_collection.find(query, {"_id": 0}))
    if not logs:
        return None, pd.DataFrame()

    df_logs = pd.DataFrame(logs)

    fig = px.bar(
        df_logs, x="Ticker", y="PredictedReturn",
        color="Signal", title="MongoDB Forecast Logs"
    )
    fig.update_layout(height=500)

    return fig, df_logs


# ======================================================
# 6 — GRADIO DASHBOARD
# ======================================================
with gr.Blocks(title="Stock Dashboard") as demo:

    gr.Markdown("## 📊 Stock Dashboard — ML Forecasting + Mongo Logging")

    # ----- TAB 1 -----
    with gr.Tab("📈 Trend Viewer"):
        tickers_tab1 = gr.Dropdown(TICKER_LIST, multiselect=True, label="Select Tickers")
        window_tab1 = gr.Radio(
            ["Daily", "Weekly", "Monthly", "Yearly", "Full History"],
            value="Daily", label="Time Window"
        )
        btn1 = gr.Button("Load Chart")
        out_plot1 = gr.Plot()
        btn1.click(load_data, [tickers_tab1, window_tab1], out_plot1)

    # ----- TAB 2 -----
    with gr.Tab("📊 BUY / HOLD / SELL"):
        tickers_tab2 = gr.Dropdown(TICKER_LIST, multiselect=True, label="Select Companies")
        horizon_tab2 = gr.Radio(
            ["Next 7 days", "Next 10 days", "Next 15 days"],
            value="Next 7 days", label="Forecast Horizon"
        )
        btn2 = gr.Button("Compare")
        out_plot2 = gr.Plot()
        out_df2 = gr.Dataframe()
        summary2 = gr.Textbox(label="Summary", lines=10)
        btn2.click(compare_companies, [tickers_tab2, horizon_tab2],
                   [out_plot2, out_df2, summary2])

    # ----- TAB 3 -----
    with gr.Tab("🗄️ MongoDB Logs"):
        ticker_filter = gr.Dropdown(["All"] + TICKER_LIST, label="Filter by Ticker")
        btn3 = gr.Button("View Logs")
        log_plot = gr.Plot()
        log_df = gr.Dataframe()
        btn3.click(view_logs, [ticker_filter], [log_plot, log_df])


# ======================================================
# 7 — LAUNCH
# ======================================================
demo.launch()
